# Training Mechanics and Evaluation

Goal: Practice model evaluation, metrics and hyperparameter tuning on medical classification.

In [ ]:
import sys
sys.path.append("../")  # adds parent folder to Python path
from workshop_utils import load_dataset, basic_train_test_split, build_preprocessor
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, roc_auc_score

df = load_dataset("cvd")
X_train, X_test, y_train, y_test = basic_train_test_split(df, target="CVD")

num_features = ["age", "chol", "bp"]
cat_features = []
preprocessor = build_preprocessor(num_features, cat_features)

base_pipe = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", MLPClassifier(max_iter=1000)),
    ]
)

param_grid = {
    "model__hidden_layer_sizes": [(32,), (64,), (32, 16)],
    "model__alpha": [0.001, 0.0001],
    "model__activation": ["identity"]
}

search = GridSearchCV(base_pipe, param_grid=param_grid, cv=5)
search.fit(X_train, y_train)
print("Best params:", search.best_params_)

y_pred = search.predict(X_test)
print(classification_report(y_test, y_pred))

if hasattr(search.best_estimator_["model"], "predict_proba"):
    y_prob = search.predict_proba(X_test)[:, 1]
    print("ROC AUC:", roc_auc_score(y_test, y_prob))

Best params: {'model__alpha': 0.0001, 'model__hidden_layer_sizes': (32,)}
              precision    recall  f1-score   support

           0       0.68      0.71      0.69        95
           1       0.73      0.70      0.71       105

    accuracy                           0.70       200
   macro avg       0.70      0.71      0.70       200
weighted avg       0.71      0.70      0.71       200

ROC AUC: 0.7661152882205514


### Notes of some hidden tools.
- Set `random_state` in MLP so results are reproducible.
- Consider `early_stopping=True` to avoid long/noisy training.
- Use use a different `scoring` rule in `GridSewarchCV` if accuracy isn’t the right metric.
- Use `stratify=y` in your train/test split to keep class balance.
- Big grids × 5-fold CV can be slow → use `n_jobs=-1`.
- The workshop lead will have no idea what the best solution is.